# Map of all compounds in the input dataset

PubChem, HMDB, NIST.

In [2]:
INPUT_FILE = "Compounds.xlsx"
INPUT_DIRECTORY = "../data/"
OUTPUT_DIRECTORY = "../results/"

In [3]:
import pandas as pd
import os
import requests
import nistchempy as nist

In [4]:
df = pd.read_excel(os.path.join(INPUT_DIRECTORY, INPUT_FILE), header=None)
df.head()

df.rename(columns={0: "Input Name"}, inplace=True)

df.insert(1, "Common/IUPAC Name", None)
df.insert(2, "InChI", None)
df.insert(3, "InChIKey", None)
df.insert(4, "CID", None)
df.insert(5, "HMDB ID", None)
df.insert(6, "CHEBI ID", None)
df.insert(7, "RI Polar (NIST)", None)
df.insert(8, "RI Polar (HMDB)", None)
df.insert(9, "Ontology Links", None)
df.insert(10, "Pathway", None)
df.insert(11, "RI Polar(HMDB)", None)
df.insert(12, "Biomarker", None)
df.insert(13, "Disease Association", None)
df.insert(14, "SMILES", None)
df.insert(15, "Taxonomy", None)
df.insert(16, "Is Terpene", None)

df.head(5)

,Input Name,Common/IUPAC Name,InChI,InChIKey,CID,HMDB ID,CHEBI ID,RI Polar (NIST),RI Polar (HMDB),Ontology Links,Pathway,RI Polar(HMDB),Biomarker,Disease Association,SMILES,Taxonomy,Is Terpene
0,(+)-Isomenthol,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,"(1R)-2,6,6-Trimethylbicyclo[3.1.1]hept-2-ene",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,(E)-Hexadec-2-enal,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,.alpha. Isomethyl ionone,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,.alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]ox...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


### PubChem Identification

By Input Name we retrieve CID and all synonyms.

In [ ]:
PUBCHEM_API_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound"

compound_synonyms = pd.DataFrame()
compound_synonyms.insert(0, "Input Name", None)
compound_synonyms.insert(1, "CID", None)
compound_synonyms.insert(2, "CHEBI ID", None)

for index, row in df.iterrows():
    compound_name = row["Input Name"]
    pubchem_synonyms_result = requests.get(f"{PUBCHEM_API_URL}/name/{compound_name}/synonyms/JSON").json()
    if "InformationList" not in pubchem_synonyms_result or len(pubchem_synonyms_result["InformationList"]["Information"]) == 0:
        print(f"No result in PubChem found for {index}: {compound_name}")
        continue
    synonyms = pubchem_synonyms_result["InformationList"]["Information"][0]["Synonym"]
    cid = pubchem_synonyms_result["InformationList"]["Information"][0]["CID"]
    compound_synonyms.at[index, "Input Name"] = compound_name
    compound_synonyms.at[index, "CID"] = cid
    df.at[index, "CID"] = cid
    for i, s in enumerate(synonyms):
        if i > 20: # limit to 20 synonyms to avoid too many columns
            break
        if i >= len(compound_synonyms.columns) - 2: # if we need more columns, add them
            compound_synonyms.insert(len(compound_synonyms.columns), f"Synonym {i}", None)
        compound_synonyms.at[index, f"Synonym {i}"] = s

By CID we retrieve InChI, InChIKey and IUPAC Name.

In [ ]:
for index, row in df.iterrows():
    cid = row["CID"]
    pubchem_result = requests.get(f"{PUBCHEM_API_URL}/cid/{cid}/property/Title,SMILES,InChI,InChIKey,IUPACName/JSON").json()
    if "PropertyTable" not in pubchem_result or len(pubchem_result["PropertyTable"]["Properties"]) == 0:
        print(f"No result in PubChem found for CID {row['Input Name']}")
        continue
    props = pubchem_result["PropertyTable"]["Properties"][0]
    df.at[index, "Common/IUPAC Name"] = props["IUPACName"]
    df.at[index, "InChI"] = props["InChI"]
    df.at[index, "InChIKey"] = props["InChIKey"]
    df.at[index, "SMILES"] = props["SMILES"]

df.head(5)

Saving synonyms dataframe to CSV.

In [ ]:
compound_synonyms.to_csv(os.path.join(OUTPUT_DIRECTORY, "compound_synonyms.csv"), index=False)
# compound_synonyms = pd.read_csv(os.path.join(OUTPUT_DIRECTORY, "compound_synonyms.csv"))
compound_synonyms.head()

By the end of the PubChem mapping we have 15 compounds not identified.

### ChEBI Identification

Checking for those that were not found in PubChem.

If found in ChEBI, we retrieve CHEBI ID, HMDB ID, Ontology Links, SMILES, InChI and InChIKey.

In [208]:
import json

CHEBI_API_URL = "https://www.ebi.ac.uk/chebi/backend/api/public/"

not_found_in_pubchem = df[df["CID"].isna()]

for id, row in not_found_in_pubchem.iterrows():
    name = row["Input Name"]
    chebi_id_request = requests.get(f"{CHEBI_API_URL}es_search/?term={name}&page=1&size=15").json()

    if chebi_id_request is None or "results" not in chebi_id_request or len(chebi_id_request["results"]) == 0:
        print(f"No result in ChEBI found for [{id}] {name}")
        continue

    print(f"Found ChEBI ID for [{id}] {name}: {chebi_id_request['results'][0]['_source']['chebi_accession']}")

    chebi_id = chebi_id_request["results"][0]["_source"]["chebi_accession"]

    chebi_request = requests.get(f"{CHEBI_API_URL}compound/{chebi_id}?only_ontology_parents=false&only_ontology_children=false").json()
    
    if chebi_request is None or "database_accessions" not in chebi_request or "MANUAL_X_REF" not in chebi_request["database_accessions"]:
        print(f"No result in ChEBI found for [{id}] {name}")
        continue

    hmdb_id = next(
        (
            x.get("accession_number")
            for x in chebi_request.get("database_accessions", {}).get("MANUAL_X_REF", [])
            if x.get("prefix") == "hmdb"
        ),
        None
    )

    ontology_relations = chebi_request["ontology_relations"]
    ontologies = {
        "incoming": [
            f"{rel['init_name']} {rel['relation_type']} {rel['final_name']}" for rel in ontology_relations["incoming_relations"]
        ],
        "outgoing": [
            f"{rel['init_name']} {rel['relation_type']} {rel['final_name']}" for rel in ontology_relations["outgoing_relations"]
        ]
    }

    df.at[id, "Common/IUPAC Name"] = chebi_request["names"]["IUPAC NAME"][0]["name"] if "IUPAC NAME" in chebi_request["names"] else None
    df.at[id, "CHEBI ID"] = chebi_id
    df.at[id, "HMDB ID"] = hmdb_id
    df.at[id, "Ontology Links"] = json.dumps(ontologies)
    df.at[id, "SMILES"] = chebi_request["default_structure"]["smiles"]
    df.at[id, "InChI"] = chebi_request["default_structure"]["standard_inchi"]
    df.at[id, "InChIKey"] = chebi_request["default_structure"]["standard_inchi_key"]

    print("\t" + "\n\t".join(f"{col}: {df.loc[id][col]}" for col in df.columns))

    # add synonyms from ChEBI to the compound_synonyms dataframe
    synonyms = chebi_request["names"]["SYNONYM"] if "SYNONYM" in chebi_request["names"] else []
    if len(synonyms) > 0:
        compound_synonyms.at[id, "Input Name"] = name
        for i, s in enumerate(synonyms):
            if i > 20: # limit to 20 synonyms to avoid too many columns
                break
            if i >= len(compound_synonyms.columns) - 2: # if we need more columns, add them
                compound_synonyms.insert(len(compound_synonyms.columns), f"Synonym {i}", None)
            compound_synonyms.at[id, f"Synonym {i}"] = s


n_found_after_chebi = len(not_found_in_pubchem) -  len(df[df["CID"].isna() & df["HMDB ID"].isna() & df["CHEBI ID"].isna()])
print(f"\n\nNumber of compounds found after ChEBI search: {n_found_after_chebi} out of {len(not_found_in_pubchem)} ({n_found_after_chebi/len(not_found_in_pubchem)*100:.2f}%)")
print(f"Remained unidentified compounds after ChEBI search: {len(df[df['CID'].isna() & df['HMDB ID'].isna() & df['CHEBI ID'].isna()])} out of {len(df)} ({len(df[df['CID'].isna() & df['HMDB ID'].isna() & df['CHEBI ID'].isna()])/len(df)*100:.2f}%)")

Found ChEBI ID for [10] 1,2-Benzenedicarboxylic acid, bis(2-methylpropyl) ester, : CHEBI:79053
	Input Name: 1,2-Benzenedicarboxylic acid, bis(2-methylpropyl) ester, 
	Common/IUPAC Name: diisobutyl phthalate
	InChI: InChI=1S/C16H22O4/c1-11(2)9-19-15(17)13-7-5-6-8-14(13)16(18)20-10-12(3)4/h5-8,11-12H,9-10H2,1-4H3
	InChIKey: MGWAVDBGNNKXQV-UHFFFAOYSA-N
	CID: None
	HMDB ID: HMDB0013835
	CHEBI ID: CHEBI:79053
	RI Polar (NIST): None
	RI Polar (HMDB): 2584.2520
	Ontology Links: {"incoming": [], "outgoing": ["diisobutyl phthalate has functional parent isobutanol", "diisobutyl phthalate has role teratogenic agent", "diisobutyl phthalate has role PPAR modulator", "diisobutyl phthalate has role plasticiser", "diisobutyl phthalate is a phthalate ester", "diisobutyl phthalate is a diester"]}
	Pathway: []
	Biomarker: []
	Disease Association: []
	SMILES: CC(C)COC(=O)c1ccccc1C(=O)OCC(C)C
	Taxonomy: {"description": null, "kingdom": "Organic compounds", "superklass": "Benzenoids", "klass": "Benzene and 

### ChemSpider Identification

In [209]:
import os
from dotenv import load_dotenv
from chemspipy import ChemSpider

load_dotenv()

CHEMSPIDER_API_KEY = os.getenv("CHEMSPIDER_API_KEY")

cs = ChemSpider("YOUR_API_KEY")

not_found = df[df["CID"].isna() & df["HMDB ID"].isna()]

for index, row in not_found.iterrows():
    name = row["Input Name"]
    # print(name)
    res = cs.search(name)
    res.wait()
    # print(res)

n_found_after_chemspider = len(not_found) -  len(df[df["CID"].isna() & df["HMDB ID"].isna() & df["CHEBI ID"].isna()])
print(f"\n\nNumber of compounds found after ChemSpider search: {n_found_after_chemspider} out of {len(not_found)} ({n_found_after_chemspider/len(not_found)*100:.2f}%)")
print(f"Remained unidentified compounds after ChemSpider search: {len(df[df['CID'].isna() & df['HMDB ID'].isna() & df['CHEBI ID'].isna()])} out of {len(df)} ({len(df[df['CID'].isna() & df['HMDB ID'].isna() & df['CHEBI ID'].isna()])/len(df)*100:.2f}%)")



Number of compounds found after ChemSpider search: 0 out of 14 (0.00%)
Remained unidentified compounds after ChemSpider search: 14 out of 408 (3.43%)


## HMDB

Retrieving HMDB ID for all identified compounds.

First by InChI, InChIKey and SMILES, then by name (all synonyms).

In [ ]:
import math
import sqlite3

not_found_hmdb_id = df[df["HMDB ID"].isna()]
print(f"Compounds without HMDB ID: {len(not_found_hmdb_id)} out of {len(df)} ({len(not_found_hmdb_id)/len(df)*100:.2f}%)")

HMDB_SQLITE_PATH = "../data/hmdb.sqlite"


def is_missing(x):
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    s = str(x).strip().lower()
    return s == "" or s in {"none", "nan", "null"}


conn = sqlite3.connect(HMDB_SQLITE_PATH)

conn.row_factory = sqlite3.Row

# Searching by InChI, InChIKey and SMILES
for index, row in df.iterrows():
    if not is_missing(row.get("HMDB ID")):
        continue

    inchi = row.get("InChI")
    inchikey = row.get("InChIKey")
    smiles = row.get("SMILES")

    cursor = conn.execute(
        """
        SELECT hmdb_id
        FROM metabolites
        WHERE inchi_key = ?
           OR inchi = ?
           OR smiles = ?
        LIMIT 1
        """,
        (inchikey, inchi, smiles)
    )
    result = cursor.fetchone()

    if result is not None:
        df.at[index, "HMDB ID"] = result["hmdb_id"]
        print(f"matched by identifier: row {index} -> {result['hmdb_id']}")


# Searching by input name and all available synonyms
for index, row in df.iterrows():
    if not is_missing(row.get("HMDB ID")):
        continue

    input_name = row.get("Input Name")
    if is_missing(input_name):
        print(f"not found for row {index}: empty Input Name")
        continue

    syn_rows = compound_synonyms.loc[
        compound_synonyms["Input Name"] == input_name
    ]

    if syn_rows.empty:
        print(f"not found for {input_name}: no synonyms row")
        continue

    found = False

    for _, syn_row in syn_rows.iterrows():
        for col in syn_row.index:
            if col == "Input Name":
                continue

            synonym = syn_row[col]
            if is_missing(synonym):
                continue

            synonym = str(synonym).strip()

            cursor = conn.execute(
                """
                SELECT hmdb_id, name, synonyms
                FROM metabolites
                WHERE lower(name) = lower(?)
                   OR lower(synonyms) LIKE lower(?)
                LIMIT 1
                """,
                (synonym, f"%{synonym}%")
            )
            result = cursor.fetchone()

            if result is not None:
                df.at[index, "HMDB ID"] = result["hmdb_id"]
                print(
                    f"matched by synonym: row {index} ({input_name}) "
                    f"using '{synonym}' -> {result['hmdb_id']}"
                )
                found = True
                break

        if found:
            break

    if not found:
        print(f"not found for {input_name}")

conn.close()

Filling in all HMDB info (chemical taxonomy (with "Is Terpene" flag), disease associations, pathways, RI, biomarker) for all compounds with HMDB ID.

In [14]:
# fill in all HMDB info
import json

conn = sqlite3.connect(HMDB_SQLITE_PATH)


for index, row in df.iterrows():
    hmdb_id = row.get("HMDB ID")
    if is_missing(hmdb_id):
        continue

    if not is_missing(row.get("RI Polar (HMDB)")):
        continue

    cursor = conn.execute(
        """
        SELECT o.metabolic_processes_json, o.diseases_json, m.chemical_taxonomy_json, m.retention_indices_json, o.biomarker_for_json
        FROM metabolites m
        JOIN ontology o ON m.hmdb_id = o.hmdb_id
        WHERE m.hmdb_id = ?
        """,
        (hmdb_id,)
    )
    result = cursor.fetchone()


    if result is not None:
        if result[3] is not None:
            data = json.loads(result[3])
            ri = next((item["retention_index"] for item in data if item["stationary_phase"] == "Standard polar"), None)

            if ri is None:
                ri = next((item["retention_index"] for item in data if item["stationary_phase"] == "Semi standard polar"), None)
        else:
            ri = None

        df.at[index, "Pathway"] = result[0]  # metabolic_processes_json
        df.at[index, "Disease Association"] = result[1]  # diseases_json
        df.at[index, "Taxonomy"] = result[2]  # chemical_taxonomy_json
        df.at[index, "Is Terpene"] = "terpen" in result[2].lower() if result[2] else None
        df.at[index, "RI Polar (HMDB)"] = ri  # retention_indices_json
        df.at[index, "Biomarker"] = result[4]  # biomarker_for_json

In [15]:
df.to_csv(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.csv"), index=False)
# df = pd.read_csv(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.csv"))

no_hmdb_id = df[df["HMDB ID"].isna()]
print(f"Compounds without HMDB ID: {len(no_hmdb_id)} out of {len(df)} ({len(no_hmdb_id) / len(df) * 100:.2f}%)")

Compounds without HMDB ID: 148 out of 408 (36.27%)


## KEGG 

In [ ]:
import requests
from urllib.parse import quote

KEGG_API_URL = "https://rest.kegg.jp/"

def kegg_get_text(endpoint: str):
    r = requests.get(f"{KEGG_API_URL}{endpoint}", timeout=30)
    if r.status_code != 200:
        return None
    text = r.text.strip()
    return text if text else None

def parse_kegg_find(text: str) -> list[dict]:
    results = []
    for line in text.splitlines():
        parts = line.split("\t", 1)
        if len(parts) != 2:
            continue
        kegg_id, names = parts
        results.append({
            "kegg_id": kegg_id.strip(),
            "names": [x.strip() for x in names.split(";")]
        })
    return results

def parse_kegg_link(text: str) -> list[str]:
    out = []
    for line in text.splitlines():
        parts = line.split("\t")
        if len(parts) == 2:
            out.append(parts[1].strip())
    return out

def parse_kegg_flat(text: str) -> dict:
    data = {}
    current_key = None

    for line in text.splitlines():
        if not line.strip():
            continue

        key = line[:12].strip()
        value = line[12:].strip()

        if key:
            current_key = key
            data.setdefault(current_key, []).append(value)
        elif current_key:
            data[current_key].append(value)

    return data

# def find_compound_in_kegg_by_name(name: str):
#     if not isinstance(name, str) or not name.strip():
#         return None
#     query = quote(name.strip(), safe="")
#     text = kegg_get_text(f"find/compound/{query}/nop")
#     if not text:
#         return None
#     return parse_kegg_find(text)

def kegg_from_chebi(chebi_id: str):
    if not chebi_id:
        return []
    chebi_id = str(chebi_id).replace("CHEBI:", "").strip()
    text = kegg_get_text(f"conv/compound/chebi:{chebi_id}")
    # if not text:
    #     return []
    # return [line.split("\t")[0].strip() for line in text.splitlines() if "\t" in line]
    return text

def kegg_from_pubchem(pubchem_id: str):
    if not pubchem_id:
        return []
    text = kegg_get_text(f"conv/compound/pubchem:{pubchem_id}")
    if not text:
        return []
    return [line.split("\t")[0].strip() for line in text.splitlines() if "\t" in line]

def get_kegg_entry(kegg_id: str):
    text = kegg_get_text(f"get/{kegg_id}")
    if not text:
        return None
    return parse_kegg_flat(text)

def get_kegg_pathways(kegg_id: str):
    text = kegg_get_text(f"link/pathway/{kegg_id}")
    return parse_kegg_link(text) if text else []

def get_kegg_diseases(kegg_id: str):
    text = kegg_get_text(f"link/disease/{kegg_id}")
    return parse_kegg_link(text) if text else []

def get_kegg_records(ids: list[str]):
    if not ids:
        return {}
    text = kegg_get_text(f"get/{'+'.join(ids)}")
    if not text:
        return {}
    chunks = text.split("///")
    out = {}
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        parsed = parse_kegg_flat(chunk)
        entry = parsed.get("ENTRY", [""])[0].split()[0]
        if entry:
            out[entry] = parsed
    return out

In [ ]:
# KEGG_API_URL = "https://rest.kegg.jp/"

# row = no_hmdb_id.iloc[0]
# compound_name = row["Input Name"]
# response = requests.get(f"{KEGG_API_URL}{compound_name}")
# print(response.status_code)

# def parse_kegg_find(text):
#     results = []

#     for line in text.strip().split("\n"):
#         parts = line.split("\t")
#         if len(parts) != 2:
#             continue

#         kegg_id, names = parts
#         results.append({
#             "kegg_id": kegg_id,
#             "names": [n.strip() for n in names.split(";")]
#         })

#     return results

def find_compound_in_kegg(name):
    response = requests.get(f"{KEGG_API_URL}find/compound/{name}/nop")
    
    if response.status_code != 200 or not response.text.strip():
        return None
    
    return parse_kegg_find(response.text)


def find_compound_in_kegg_by_name(name):
    kegg_response = find_compound_in_kegg(name)

    if kegg_response is None:
        synonyms = compound_synonyms[compound_synonyms["Input Name"] == compound_name]
        if not synonyms.empty:
            for col in synonyms.columns:
                if col == "Input Name":
                    continue
                synonym = synonyms.iloc[0][col]
                if is_missing(synonym):
                    continue
                kegg_response = find_compound_in_kegg(synonym)
                if kegg_response is not None:
                    break

    if kegg_response is None:
        return None

    print(kegg_response)
    return kegg_response

found = 0

for index, row in no_hmdb_id.iterrows():
    compound_name = row["Input Name"]
    chebi_id = row.get("ChEBI ID")
    pubchem_id = row.get("CID")

    kegg_ids = []
    by_cid = True

    if not kegg_ids and pd.notna(pubchem_id):
        kegg_ids = kegg_from_pubchem(pubchem_id)

    if not kegg_ids:
        print(f"KEGG not found for {compound_name}")
        continue

    print(f"Found KEGG ID for {compound_name} (by {'cid' if by_cid else 'name'}): \n\t{kegg_ids}")

    found += 1

print(f"\nFound KEGG IDs for {found} compounds")

print(f"\n\nNumber of compounds found in KEGG: {found} out of {len(no_hmdb_id)} ({found/len(no_hmdb_id)*100:.2f}%)")

KEGG not found for (+)-Isomenthol
KEGG not found for .alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]oxiranemethanol
KEGG not found for 1,1'-Biphenyl, 2,2',5,5'-tetramethyl-
KEGG not found for 1,3-di-iso-propylnaphthalene
KEGG not found for 1,4-Benzenedicarboxaldehyde
KEGG not found for 1,4-Benzenedicarboxylic acid, dimethyl ester
KEGG not found for 1,4-Dihydronaphthalene
KEGG not found for 1,4-Methanoazulen-9-ol, decahydro-1,5,5,8a-tetramethyl-, [1R-(1.alpha.,3a.beta.,4.alpha.,8a.beta.,9S*)]-
KEGG not found for 1,4-Methanobenzocyclodecene, 1,2,3,4,4a,5,8,9,12,12a-decahydro-
KEGG not found for 1,4-di-iso-propylnaphthalene
KEGG not found for 1,6-Dioxacyclododecane-7,12-dione
KEGG not found for 1,7-di-iso-propylnaphthalene
KEGG not found for 1-(4-tert-Butylphenyl)propan-2-one
KEGG not found for 1-Nonadecene
KEGG not found for 11H-Dibenzo[b,e][1,4]diazepin-11-one, 5-(3-aminopropyl)-5,10-dihydro-
KEGG not found for 1H-Inaene, 2,3-dihydro-2-methyl-
KEGG not found for 1H-Naphtho[2,1-b]pyran, 3-e

## ClassyFire

In [16]:
import json

CLASSYFIRE_API_URL = "http://classyfire.wishartlab.com/entities/{}.json"

def transform_taxonomy(taxonomy_json):
    return {
        "description": None,
        "kingdom": taxonomy_json.get("kingdom", {}).get("name"),
        "superklass": taxonomy_json.get("superclass", {}).get("name"),
        "klass": taxonomy_json.get("class", {}).get("name"),
        "subklass": taxonomy_json.get("subclass", {}).get("name") if taxonomy_json.get("subclass") else None,
        "direct_parent": taxonomy_json.get("direct_parent", {}).get("name"),
        "alternative_parents": [x.get("name") for x in taxonomy_json.get("alternative_parents", [])],
        "substituents": taxonomy_json.get("substituents", []),
        "molecular_framework": taxonomy_json.get("molecular_framework")
    }

found_with_classyfire = 0

for index, row in no_hmdb_id.iterrows():
    inchi_key = row["InChIKey"]
    response = requests.get(CLASSYFIRE_API_URL.format(inchi_key))

    if response.status_code == 200:
        print(f"ClassyFire found for {inchi_key}")
        data = response.json()
        taxonomy = transform_taxonomy(data)
        print(f"\t{taxonomy}")
        taxonomy_str = json.dumps(taxonomy)
        df.at[index, "Taxonomy"] = taxonomy_str
        df.at[index, "Is Terpene"] = "terpen" in taxonomy_str.lower()
        found_with_classyfire += 1
    else:
        print(f"ClassyFire not found for {inchi_key}")

print(f"\n\nIdentified with ClassyFire: {found_with_classyfire} out of {len(no_hmdb_id)} ({found_with_classyfire/len(no_hmdb_id)*100:.2f}%)")
print(f"Taxonomies filled for {len(df[~df['Taxonomy'].isna()])} compounds out of {len(df)} ({len(df[~df['Taxonomy'].isna()])/len(df)*100:.2f}%)")
print(f"Missing taxonomies for {len(df[df['Taxonomy'].isna()])} compounds out of {len(df)} ({len(df[df['Taxonomy'].isna()])/len(df)*100:.2f}%)")

ClassyFire found for NOOLISFMXDJSKH-BBBLOLIVSA-N
	{'description': None, 'kingdom': 'Organic compounds', 'superklass': 'Lipids and lipid-like molecules', 'klass': 'Prenol lipids', 'subklass': 'Monoterpenoids', 'direct_parent': 'Menthane monoterpenoids', 'alternative_parents': ['Monocyclic monoterpenoids', 'Cyclohexanols', 'Cyclic alcohols and derivatives', 'Hydrocarbon derivatives'], 'substituents': ['P-menthane monoterpenoid', 'Monocyclic monoterpenoid', 'Cyclohexanol', 'Cyclic alcohol', 'Secondary alcohol', 'Organic oxygen compound', 'Hydrocarbon derivative', 'Organooxygen compound', 'Alcohol', 'Aliphatic homomonocyclic compound'], 'molecular_framework': 'Aliphatic homomonocyclic compounds'}
ClassyFire found for BXOURKNXQXLKRK-UHFFFAOYSA-N
	{'description': None, 'kingdom': 'Organic compounds', 'superklass': 'Organic oxygen compounds', 'klass': 'Organooxygen compounds', 'subklass': 'Alcohols and polyols', 'direct_parent': 'Tertiary alcohols', 'alternative_parents': ['Oxacyclic compound

## NIST Chemistry WebBook

By InChi, retrieving retention index.

In [30]:
for index, row in df.iterrows():
    inchi = row["InChI"]
    if pd.isna(inchi):
        print(f"No InChI for {row['Input Name']}, skipping NIST lookup")
        continue

    nist_compound = nist.get_compound(inchi)

    if nist_compound is None:
        print(f"NIST compound not found for {row['Input Name']} with InChI {inchi}")
        continue
    
    nist_compound.get_gas_chromatography()

    gc_all = nist_compound.gas_chromat
    gc_filtered = [gc for gc in gc_all if gc.ri_type == "Normal alkane RI" and gc.column_type == "polar column"]

    means = []
    for gc in gc_filtered:
        average_ri = gc.data[["I"]].astype(float).mean()["I"]
        means.append(average_ri)
    ri = sum(means) / len(means) if means else None
    df.at[index, "RI Polar (NIST)"] = ri
    print(f"NIST RI for {row['Input Name']}: {ri}")

print(f"\n\nFilled RI Polar (NIST) for {len(df[~df['RI Polar (NIST)'].isna()])} compounds out of {len(df)} ({len(df[~df['RI Polar (NIST)'].isna()])/len(df)*100:.2f}%)")

I    1668.75
dtype: float64
I    1667.0
dtype: float64
NIST RI for (+)-Isomenthol: 1667.875
NIST compound not found for (1R)-2,6,6-Trimethylbicyclo[3.1.1]hept-2-ene with InChI InChI=1S/C10H16/c1-7-4-5-8-6-9(7)10(8,2)3/h4,8-9H,5-6H2,1-3H3/t8-,9-/m1/s1
NIST compound not found for (E)-Hexadec-2-enal with InChI InChI=1S/C16H30O/c1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17/h14-16H,2-13H2,1H3/b15-14+
NIST compound not found for .alpha. Isomethyl ionone with InChI InChI=1S/C14H22O/c1-10-7-6-8-14(4,5)13(10)9-11(2)12(3)15/h7,9,13H,6,8H2,1-5H3/b11-9+
NIST compound not found for .alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]oxiranemethanol with InChI InChI=1S/C10H18O2/c1-8(2)5-4-6-10(3,11)9-7-12-9/h5,9,11H,4,6-7H2,1-3H3
NIST compound not found for .alpha.-Terpineol with InChI InChI=1S/C10H18O/c1-8-4-6-9(7-5-8)10(2,3)11/h4,9,11H,5-7H2,1-3H3
I    1161.304545
dtype: float64
I    1164.972222
dtype: float64
NIST RI for .beta.-Myrcene: 1163.1383838383838
NIST compound not found for .beta.-iso-Methyl ionone

## MarkerDB

Retrieving biomarker information for all compounds without HMDB ID.

In [54]:
from dotenv import load_dotenv

load_dotenv()

MARKERDB_API_KEY = "cc69fb7c1a4809bbae7c0701d5188c17"

MARKERDB_API_URL = "http://markerdb.ca/api/v1/chemicalapi/chemicalrequest"

found = 0

for index, row in no_hmdb_id.iterrows():
    input_name = row["Input Name"]
    name = input_name.replace("-", " ").replace("'", "").strip()

    response = requests.get(MARKERDB_API_URL, {
        "name": name,
        "api_key": MARKERDB_API_KEY,
        "page": 1,
    }).json()

    if len(response['chemicals']) == 0:
        synonyms = compound_synonyms[compound_synonyms["Input Name"] == input_name]
        if not synonyms.empty:
            for col in synonyms.columns:
                if col == "Input Name":
                    continue
                synonym = synonyms.iloc[0][col]
                if is_missing(synonym):
                    continue
                synonym = str(synonym).replace("-", " ").replace("'", "").strip()
                response = requests.get(MARKERDB_API_URL, {
                    "name": synonym,
                    "api_key": MARKERDB_API_KEY,
                    "page": 1,
                }).json()

                if len(response['chemicals']) > 0 and response['chemicals'] != '[]':
                    break
    
    if len(response['chemicals']) == 0 or response['chemicals'] == '[]':
        print(f"MarkerDB not found for {input_name}")
        continue
    print(f"MarkerDB found for {input_name}: {response['chemicals']}")
    found += 1

print(f"\n\nFound in MarkerDB: {found} out of {len(no_hmdb_id)} ({found/len(no_hmdb_id)*100:.2f}%)")

MarkerDB not found for (+)-Isomenthol
MarkerDB not found for .alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]oxiranemethanol
MarkerDB not found for 1,1'-Biphenyl, 2,2',5,5'-tetramethyl-
MarkerDB not found for 1,3-di-iso-propylnaphthalene
MarkerDB not found for 1,4-Benzenedicarboxaldehyde
MarkerDB not found for 1,4-Benzenedicarboxylic acid, dimethyl ester
MarkerDB not found for 1,4-Dihydronaphthalene
MarkerDB not found for 1,4-Methanoazulen-9-ol, decahydro-1,5,5,8a-tetramethyl-, [1R-(1.alpha.,3a.beta.,4.alpha.,8a.beta.,9S*)]-
MarkerDB not found for 1,4-Methanobenzocyclodecene, 1,2,3,4,4a,5,8,9,12,12a-decahydro-
MarkerDB not found for 1,4-di-iso-propylnaphthalene
MarkerDB not found for 1,6-Dioxacyclododecane-7,12-dione
MarkerDB not found for 1,7-di-iso-propylnaphthalene
MarkerDB not found for 1-(4-tert-Butylphenyl)propan-2-one
MarkerDB not found for 1-Nonadecene
MarkerDB not found for 11H-Dibenzo[b,e][1,4]diazepin-11-one, 5-(3-aminopropyl)-5,10-dihydro-
MarkerDB not found for 1H-Inaene, 2,3-

## PathBank

Retrieving pathway information for all compounds without HMDB ID.

(As i understood, doesnt make much sense as only HMDB compounds are in PathBank)

In [31]:
pb_metabolites = pd.read_csv(os.path.join(INPUT_DIRECTORY, "pathbank_all_metabolites.csv"))
pb_pathways = pd.read_csv(os.path.join(INPUT_DIRECTORY, "pathbank_pathways.csv"))

for index, row in no_hmdb_id.iterrows():
    inchi_key = row["InChIKey"]
    inchi = row["InChI"]
    smiles = row["SMILES"]

    # get from pb_metabolites where InChIKey or InChI or SMILES matches
    pb_metabolite = pb_metabolites[
        (pb_metabolites["InChI Key"] == inchi_key) |
        (pb_metabolites["InChI"] == inchi) |
        (pb_metabolites["SMILES"] == smiles)
    ]
    if pb_metabolite.empty:
        print(f"PathBank metabolite not found for {row['Input Name']}")
        continue

    pb_id = pb_metabolite.iloc[0]["PathBank ID"]
    pb_pathway = pb_pathways[pb_pathways["SMPDB ID"] == pb_id]

    if pb_pathway.empty:
        print(f"PathBank pathway not found for {row['Input Name']} with PathBank ID {pb_id}")
        continue

    df.at[index, "Pathway"] = pb_pathway.iloc[0]["Description"]
    print(f"PathBank pathway for {row['Input Name']}: {pb_pathway.iloc[0]['Description']}")    


print(f"\n\nFilled Pathway with PathBank for {len(df[~df['Pathway'].isna() & df['HMDB ID'].isna()])} compounds out of {len(no_hmdb_id)} ({len(df[~df['Pathway'].isna() & df['HMDB ID'].isna()]) / len(no_hmdb_id) * 100:.2f}%)")
print(f"Total missing Pathway for {len(df[df['Pathway'].isna()])} compounds out of {len(df)} ({len(df[df['Pathway'].isna()]) / len(df) * 100:.2f}%)")

PathBank metabolite not found for (+)-Isomenthol
PathBank metabolite not found for .alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]oxiranemethanol
PathBank metabolite not found for 1,1'-Biphenyl, 2,2',5,5'-tetramethyl-
PathBank metabolite not found for 1,3-di-iso-propylnaphthalene
PathBank metabolite not found for 1,4-Benzenedicarboxaldehyde
PathBank metabolite not found for 1,4-Benzenedicarboxylic acid, dimethyl ester
PathBank metabolite not found for 1,4-Dihydronaphthalene
PathBank metabolite not found for 1,4-Methanoazulen-9-ol, decahydro-1,5,5,8a-tetramethyl-, [1R-(1.alpha.,3a.beta.,4.alpha.,8a.beta.,9S*)]-
PathBank metabolite not found for 1,4-Methanobenzocyclodecene, 1,2,3,4,4a,5,8,9,12,12a-decahydro-
PathBank metabolite not found for 1,4-di-iso-propylnaphthalene
PathBank metabolite not found for 1,6-Dioxacyclododecane-7,12-dione
PathBank metabolite not found for 1,7-di-iso-propylnaphthalene
PathBank metabolite not found for 1-(4-tert-Butylphenyl)propan-2-one
PathBank metabolite not 

# CHECK

In [ ]:
# df.to_csv(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.csv"), index=False)
df.to_excel(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.xlsx"), index=False)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# df

non_identified = df[
    (df["HMDB ID"].isna() & df["CID"].isna() & df["CHEBI ID"].isna())
]

without_hmdb = df[df["HMDB ID"].isna()]

print(
    f"\n\nNon identified compounds: {len(non_identified)} out of {len(df)} "
    f"({len(non_identified) / len(df) * 100:.2f}%)"
)
for name in non_identified["Input Name"]:
    print(f"\t{name}")

print(
    f"\n\nCompounds without HMDB ID: {len(without_hmdb)} out of {len(df)} "
    f"({len(without_hmdb) / len(df) * 100:.2f}%)"
)
for name in without_hmdb["Input Name"]:
    print(f"\t{name}")



Non identified compounds: 14 out of 408 (3.43%)
	1H-Inaene, 2,3-dihydro-2-methyl-
	2-Buten-1-ol, 2-ethyl-4-(2,2,3-trimethyl-3-cyclopenten-1-yl)-
	4,7-Methano-1H-inaene, octahydro-2-methylene-
	4,7-methano-1H-inaen-5-ol, 3a,4,5,6,7,7a-hexahydro-, acetate
	Benzene, (1-ethylunaecyl)-
	Benzene, (1-methylunaecyl)-
	Carbonic acid, decyl unaecyl ester
	Cyclobuta[1,2:3,4]dicyclooctene, 1,2,5,6,6a,6b,7,8,11,12,12a,12b-dodecahydro-, (6a.alpha.,6b.beta.,12a.alpha.,12b.beta.)-
	Cyclohexanol, 5-methyl-2-(1-methylethyl)-, (1.alpha.,2.beta.,5.alpha.)-(.+/-.)-
	Cyclohexanol, 5-methyl-2-(1-methylethyl)-, (1α,2β,5α)-
	Unaecanal
	Unaecanal, 2-methyl-
	Unaecane
	Unaecane, 3-methyl-


Compounds without HMDB ID: 148 out of 408 (36.27%)
	(+)-Isomenthol
	.alpha.-Methyl-.alpha.-[4-methyl-3-pentenyl]oxiranemethanol
	1,1'-Biphenyl, 2,2',5,5'-tetramethyl-
	1,3-di-iso-propylnaphthalene
	1,4-Benzenedicarboxaldehyde
	1,4-Benzenedicarboxylic acid, dimethyl ester
	1,4-Dihydronaphthalene
	1,4-Methanoazulen-9-ol, deca

/var/folders/_p/_3h1z74x7gq86x4fd8mfqz9h0000gn/T/ipykernel_68217/462032699.py:2: UserWarning: Cell contents too long (1397092), truncated to 32767 characters
  df.to_excel(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.xlsx"), index=False)
/var/folders/_p/_3h1z74x7gq86x4fd8mfqz9h0000gn/T/ipykernel_68217/462032699.py:2: UserWarning: Cell contents too long (217996), truncated to 32767 characters
  df.to_excel(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.xlsx"), index=False)
/var/folders/_p/_3h1z74x7gq86x4fd8mfqz9h0000gn/T/ipykernel_68217/462032699.py:2: UserWarning: Cell contents too long (446249), truncated to 32767 characters
  df.to_excel(os.path.join(OUTPUT_DIRECTORY, "compound_mapping.xlsx"), index=False)
